⚠️ This notebook is for experimentation and analysis only.
Production logic is implemented under `src/` and `pipeline/`.


In [1]:
import pandas as pd
from pathlib import Path

In [2]:
BASELINE_PATH = Path("../data/processed/baselines")
ML_PATH = Path("../data/processed/ml_results")

In [3]:
baseline_metrics = pd.read_csv(BASELINE_PATH / "baseline_metrics.csv")
classical_metrics = pd.read_csv(BASELINE_PATH / "classical_baseline_metrics.csv")
ml_metrics = pd.read_csv(ML_PATH / "ml_model_comparison_metrics.csv")

In [4]:
baseline_metrics["category"] = "Moving Average Baseline"
classical_metrics["category"] = "Classical Time-Series"
ml_metrics["category"] = "Machine Learning"


In [5]:
final_results = pd.concat(
    [baseline_metrics, classical_metrics, ml_metrics],
    ignore_index=True
)


In [6]:
final_results = final_results.sort_values("MAE").reset_index(drop=True)
final_results


,model,MAE,RMSE,category
0,ARIMA,0.758803,1.463391,Classical Time-Series
1,ETS,0.802044,1.486086,Classical Time-Series
2,SARIMA,0.804529,1.483037,Classical Time-Series
3,LinearRegression,1.105978,2.493875,Machine Learning
4,Ridge,1.105978,2.493875,Machine Learning
5,BestLightGBM,1.109974,2.473644,Machine Learning
6,BestXGBoost,1.116479,2.532847,Machine Learning
7,BestRandomForest,1.116625,2.475755,Machine Learning
8,LightGBM,1.116974,2.488075,Machine Learning
9,XGBoost,1.121194,2.533151,Machine Learning


In [7]:
final_results.to_csv(
    "../data/processed/final_model_comparison.csv",
    index=False
)

In [8]:
final_model = (
    final_results
    .query("category == 'Machine Learning'")
    .sort_values("RMSE")
    .iloc[0]
)

final_model


model           BestLightGBM
MAE                 1.109974
RMSE                2.473644
category    Machine Learning
Name: 5, dtype: object

## Final Model Selection – LightGBM

Although classical time-series models (ARIMA, SARIMA, ETS) achieved the lowest
error metrics, they were evaluated on a limited subset of items and require
one model per time series, making them unsuitable for large-scale deployment.

Model selection was therefore restricted to scalable machine learning approaches.
Among these, LightGBM achieved the lowest RMSE while maintaining competitive MAE,
indicating superior robustness to demand spikes and extreme values.

Given the operational cost of large forecasting errors in retail, RMSE was
prioritized during final model selection. As a result, LightGBM was chosen as
the final model due to its balance of accuracy, scalability, and robustness.


## Business Interpretation

Accurate demand forecasting is critical for inventory planning, reducing stockouts,
and minimizing excess inventory. While classical models perform well on individual
series, they lack scalability in real-world retail environments.

The selected LightGBM model minimizes large forecasting errors and generalizes
across thousands of products, enabling more reliable, data-driven operational
decisions.


## Limitations

- Classical time-series models were evaluated on a limited subset of items
- External drivers such as pricing, promotions, and competitor behavior were not available


## Conclusion

This project demonstrates that strong feature engineering combined with robust
machine learning models can significantly improve retail demand forecasting.
By systematically evaluating statistical, classical, and machine learning models,
LightGBM emerged as the most practical solution for real-world deployment.

The modular and reproducible pipeline developed in this project can be readily
extended to large-scale production forecasting systems.
